# Исследование надежности заемщиков


**Описание проекта**

Заказчик — кредитный отдел банка. Входные данные от банка — статистика о платёжеспособности клиентов.
Результаты исследования будут учтены при построении модели кредитного скоринга — специальной системы, которая оценивает способность потенциального заёмщика вернуть кредит банку.
    
    
**Цель проекта** - выявить наличия зависимости таких показателей, как семейное положение и количество детей клиента, уровень дохода и цели кредита на факт погашения кредита в срок.
    

**Описание данных**
* children — количество детей в семье
* days_employed — общий трудовой стаж в днях
* dob_years — возраст клиента в годах
* education — уровень образования клиента
* education_id — идентификатор уровня образования
* family_status — семейное положение
* family_status_id — идентификатор семейного положения
* gender — пол клиента
* income_type — тип занятости
* debt — имел ли задолженность по возврату кредитов
* total_income — ежемесячный доход
* purpose — цель получения кредита
    
**План работы**
    
1.  Предобработка данных 
    * удаление пропусков 
    * обработка аномальных значений и дубликатов
    * катекоризация данных
    

2. Исследование данных
    * зависимость между количеством детей и возвратом кредита в срок
    * зависимость между семейным положением и возвратом кредита в срок
    * зависимость между уровнем дохода и возвратом кредита в срок
    * зависимость между целями кредита и его возвратом в срок
    
    
3. Вывод 
</div>

Во второй части проекта вы выполните шаги 3 и 4. Их вручную проверит ревьюер.
Чтобы вам не пришлось писать код заново для шагов 1 и 2, мы добавили авторские решения в ячейки с кодом. 



## Откройте таблицу и изучите общую информацию о данных

**Задание 1. Импортируйте библиотеку pandas. Считайте данные из csv-файла в датафрейм и сохраните в переменную `data`. Путь к файлу:**

`/datasets/data.csv`

In [1]:
import pandas as pd

try:
    data = pd.read_csv('/datasets/data.csv')
except:
    data = pd.read_csv('https://code.s3.yandex.net/datasets/data.csv')

**Задание 2. Выведите первые 20 строчек датафрейма `data` на экран.**

In [2]:
data.head(20)

,children,days_employed,dob_years,education,education_id,family_status,family_status_id,gender,income_type,debt,total_income,purpose
0,1,-8437.673028,42,высшее,0,женат / замужем,0,F,сотрудник,0,253875.639453,покупка жилья
1,1,-4024.803754,36,среднее,1,женат / замужем,0,F,сотрудник,0,112080.014102,приобретение автомобиля
2,0,-5623.422610,33,Среднее,1,женат / замужем,0,M,сотрудник,0,145885.952297,покупка жилья
3,3,-4124.747207,32,среднее,1,женат / замужем,0,M,сотрудник,0,267628.550329,дополнительное образование
4,0,340266.072047,53,среднее,1,гражданский брак,1,F,пенсионер,0,158616.077870,сыграть свадьбу
5,0,-926.185831,27,высшее,0,гражданский брак,1,M,компаньон,0,255763.565419,покупка жилья
6,0,-2879.202052,43,высшее,0,женат / замужем,0,F,компаньон,0,240525.971920,операции с жильем
7,0,-152.779569,50,СРЕДНЕЕ,1,женат / замужем,0,M,сотрудник,0,135823.934197,образование
8,2,-6929.865299,35,ВЫСШЕЕ,0,гражданский брак,1,F,сотрудник,0,95856.832424,на проведение свадьбы
9,0,-2188.756445,41,среднее,1,женат / замужем,0,M,сотрудник,0,144425.938277,покупка жилья для семьи


**Задание 3. Выведите основную информацию о датафрейме с помощью метода `info()`.**

In [3]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21525 entries, 0 to 21524
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   children          21525 non-null  int64  
 1   days_employed     19351 non-null  float64
 2   dob_years         21525 non-null  int64  
 3   education         21525 non-null  object 
 4   education_id      21525 non-null  int64  
 5   family_status     21525 non-null  object 
 6   family_status_id  21525 non-null  int64  
 7   gender            21525 non-null  object 
 8   income_type       21525 non-null  object 
 9   debt              21525 non-null  int64  
 10  total_income      19351 non-null  float64
 11  purpose           21525 non-null  object 
dtypes: float64(2), int64(5), object(5)
memory usage: 2.0+ MB


## Предобработка данных

### Удаление пропусков

**Задание 4. Выведите количество пропущенных значений для каждого столбца. Используйте комбинацию двух методов.**

In [4]:
data.isna().sum()

children               0
days_employed       2174
dob_years              0
education              0
education_id           0
family_status          0
family_status_id       0
gender                 0
income_type            0
debt                   0
total_income        2174
purpose                0
dtype: int64

**Задание 5. В двух столбцах есть пропущенные значения. Один из них — `days_employed`. Пропуски в этом столбце вы обработаете на следующем этапе. Другой столбец с пропущенными значениями — `total_income` — хранит данные о доходах. На сумму дохода сильнее всего влияет тип занятости, поэтому заполнить пропуски в этом столбце нужно медианным значением по каждому типу из столбца `income_type`. Например, у человека с типом занятости `сотрудник` пропуск в столбце `total_income` должен быть заполнен медианным доходом среди всех записей с тем же типом.**

In [5]:
for t in data['income_type'].unique():
    data.loc[(data['income_type'] == t) & (data['total_income'].isna()), 'total_income'] = \
    data.loc[(data['income_type'] == t), 'total_income'].median()

### Обработка аномальных значений

**Задание 6. В данных могут встречаться артефакты (аномалии) — значения, которые не отражают действительность и появились по какой-то ошибке. таким артефактом будет отрицательное количество дней трудового стажа в столбце `days_employed`. Для реальных данных это нормально. Обработайте значения в этом столбце: замените все отрицательные значения положительными с помощью метода `abs()`.**

In [6]:
data['days_employed'] = data['days_employed'].abs()

**Задание 7. Для каждого типа занятости выведите медианное значение трудового стажа `days_employed` в днях.**

In [7]:
data.groupby('income_type')['days_employed'].agg('median')

income_type
безработный        366413.652744
в декрете            3296.759962
госслужащий          2689.368353
компаньон            1547.382223
пенсионер          365213.306266
предприниматель       520.848083
сотрудник            1574.202821
студент               578.751554
Name: days_employed, dtype: float64

У двух типов (безработные и пенсионеры) получатся аномально большие значения. Исправить такие значения сложно, поэтому оставьте их как есть. Тем более этот столбец не понадобится вам для исследования.

**Задание 8. Выведите перечень уникальных значений столбца `children`.**

In [8]:
data['children'].unique()

array([ 1,  0,  3,  2, -1,  4, 20,  5])

**Задание 9. В столбце `children` есть два аномальных значения. Удалите строки, в которых встречаются такие аномальные значения из датафрейма `data`.**

In [9]:
data = data[(data['children'] != -1) & (data['children'] != 20)]

**Задание 10. Ещё раз выведите перечень уникальных значений столбца `children`, чтобы убедиться, что артефакты удалены.**

In [10]:
data['children'].unique()

array([1, 0, 3, 2, 4, 5])

### Удаление пропусков (продолжение)

**Задание 11. Заполните пропуски в столбце `days_employed` медианными значениями по каждого типа занятости `income_type`.**

In [11]:
for t in data['income_type'].unique():
    data.loc[(data['income_type'] == t) & (data['days_employed'].isna()), 'days_employed'] = \
    data.loc[(data['income_type'] == t), 'days_employed'].median()

**Задание 12. Убедитесь, что все пропуски заполнены. Проверьте себя и ещё раз выведите количество пропущенных значений для каждого столбца с помощью двух методов.**

In [12]:
data.isna().sum()

children            0
days_employed       0
dob_years           0
education           0
education_id        0
family_status       0
family_status_id    0
gender              0
income_type         0
debt                0
total_income        0
purpose             0
dtype: int64

### Изменение типов данных

**Задание 13. Замените вещественный тип данных в столбце `total_income` на целочисленный с помощью метода `astype()`.**

In [13]:
data['total_income'] = data['total_income'].astype(int)

### Обработка дубликатов

**Задание 14. Обработайте неявные дубликаты в столбце `education`. В этом столбце есть одни и те же значения, но записанные по-разному: с использованием заглавных и строчных букв. Приведите их к нижнему регистру. Проверьте остальные столбцы.**

In [14]:
data['education'] = data['education'].str.lower()

**Задание 15. Выведите на экран количество строк-дубликатов в данных. Если такие строки присутствуют, удалите их.**

In [15]:
data.duplicated().sum()

71

In [16]:
data = data.drop_duplicates()

### Категоризация данных

**Задание 16. На основании диапазонов, указанных ниже, создайте в датафрейме `data` столбец `total_income_category` с категориями:**

- 0–30000 — `'E'`;
- 30001–50000 — `'D'`;
- 50001–200000 — `'C'`;
- 200001–1000000 — `'B'`;
- 1000001 и выше — `'A'`.


**Например, кредитополучателю с доходом 25000 нужно назначить категорию `'E'`, а клиенту, получающему 235000, — `'B'`. Используйте собственную функцию с именем `categorize_income()` и метод `apply()`.**

In [17]:
def categorize_income(income):
    try:
        if 0 <= income <= 30000:
            return 'E'
        elif 30001 <= income <= 50000:
            return 'D'
        elif 50001 <= income <= 200000:
            return 'C'
        elif 200001 <= income <= 1000000:
            return 'B'
        elif income >= 1000001:
            return 'A'
    except:
        pass

In [18]:
data['total_income_category'] = data['total_income'].apply(categorize_income)

**Задание 17. Выведите на экран перечень уникальных целей взятия кредита из столбца `purpose`.**

In [19]:
data['purpose'].unique()

array(['покупка жилья', 'приобретение автомобиля',
       'дополнительное образование', 'сыграть свадьбу',
       'операции с жильем', 'образование', 'на проведение свадьбы',
       'покупка жилья для семьи', 'покупка недвижимости',
       'покупка коммерческой недвижимости', 'покупка жилой недвижимости',
       'строительство собственной недвижимости', 'недвижимость',
       'строительство недвижимости', 'на покупку подержанного автомобиля',
       'на покупку своего автомобиля',
       'операции с коммерческой недвижимостью',
       'строительство жилой недвижимости', 'жилье',
       'операции со своей недвижимостью', 'автомобили',
       'заняться образованием', 'сделка с подержанным автомобилем',
       'получение образования', 'автомобиль', 'свадьба',
       'получение дополнительного образования', 'покупка своего жилья',
       'операции с недвижимостью', 'получение высшего образования',
       'свой автомобиль', 'сделка с автомобилем',
       'профильное образование', 'высшее об

**Задание 18. Создайте функцию, которая на основании данных из столбца `purpose` сформирует новый столбец `purpose_category`, в который войдут следующие категории:**

- `'операции с автомобилем'`,
- `'операции с недвижимостью'`,
- `'проведение свадьбы'`,
- `'получение образования'`.

**Например, если в столбце `purpose` находится подстрока `'на покупку автомобиля'`, то в столбце `purpose_category` должна появиться строка `'операции с автомобилем'`.**

**Используйте собственную функцию с именем `categorize_purpose()` и метод `apply()`. Изучите данные в столбце `purpose` и определите, какие подстроки помогут вам правильно определить категорию.**

In [20]:
def categorize_purpose(row):
    try:
        if 'автом' in row:
            return 'операции с автомобилем'
        elif 'жил' in row or 'недвиж' in row:
            return 'операции с недвижимостью'
        elif 'свад' in row:
            return 'проведение свадьбы'
        elif 'образов' in row:
            return 'получение образования'
    except:
        return 'нет категории'

In [21]:
data['purpose_category'] = data['purpose'].apply(categorize_purpose)

### Шаг 3. Исследуйте данные и ответьте на вопросы

#### 3.1 Есть ли зависимость между количеством детей и возвратом кредита в срок?

Сначала смотрим общее кол-во данных в столбце 'children', для этого используем метод группировки. 

In [22]:
child_table = data.groupby('children')['debt'].count()
child_table

children
0    14091
1     4808
2     2052
3      330
4       41
5        9
Name: debt, dtype: int64

Видно, что кол-во показателей в данных, где больше двух детей не очень много. Поэтому их можно
объеденить в одну категорию.

In [23]:
def children_category(children):
    if children == 1:
        return '1 ребенок'
    if children == 2:
        return '2 ребенка'
    if children >= 3:
        return 'Много детей'
    return 'Нет детей'

data['children_category'] = data['children'].apply(children_category)
data['children_category'].value_counts()

Нет детей      14091
1 ребенок       4808
2 ребенка       2052
Много детей      380
Name: children_category, dtype: int64

Построим сводную таблицу и посчитаем отношение в процентах.

In [24]:
child_depence = data.pivot_table(index=['children_category'], values='debt', aggfunc=['sum', 'count'])

In [25]:
child_depence = pd.DataFrame()
child_depence['sum'] = data.groupby('children_category')['debt'].sum()
child_depence['count'] = data.groupby('children_category')['debt'].count()
child_depence['conversion per'] = child_depence['sum'] / child_depence['count'] *100 
child_depence.sort_values('conversion per')

,sum,count,conversion per
children_category,,,
Нет детей,1063,14091,7.543822
Много детей,31,380,8.157895
1 ребенок,444,4808,9.234609
2 ребенка,194,2052,9.454191


**Вывод:** 

У клиентов без детей немного меньше вероятность стать должниками, чем у клиентов с детьми. Самая большая вероятность наличия задолженности по возврату кредитов у клиентов с 1 и 2 детьми.

#### 3.2 Есть ли зависимость между семейным положением и возвратом кредита в срок?

Построим сводную таблицу и посчитаем отношение в процентах.

In [26]:
family_pivot = data.pivot_table(index=['family_status'], values=["debt"], aggfunc=['sum', 'count'])

In [27]:
family_pivot = pd.DataFrame()
family_pivot['sum'] = data.groupby('family_status')['debt'].sum()
family_pivot['count'] = data.groupby('family_status')['debt'].count()
family_pivot['conversion per'] = family_pivot['sum'] / family_pivot['count'] *100 
family_pivot.sort_values('conversion per')

,sum,count,conversion per
family_status,,,
вдовец / вдова,63,951,6.624606
в разводе,84,1189,7.064760
женат / замужем,927,12261,7.560558
гражданский брак,385,4134,9.313014
Не женат / не замужем,273,2796,9.763948


**Вывод:** 

Их таблицы видно, что вдова/вдовец и клиенты в разводе имеет наименьшую возможность стать задолжником по кредиту. А самой опасной в этом ключе категория клиентов - неженатые и состоящие в гражданском браке.

#### 3.3 Есть ли зависимость между уровнем дохода и возвратом кредита в срок?

Смотрим общее кол-во данных в столбце 'total_income_category', используем метод группировки

In [28]:
income_table = data.groupby('total_income_category')['debt'].count()
income_table

total_income_category
A       25
B     5014
C    15921
D      349
E       22
Name: debt, dtype: int64

Из таблицы видим, что кол-во клиентов в категориях А, D и Е намного меньше, чем в B и C. 
Поэтому можем провести категоризацию по высоку/среднему/низкому доходу.
Будем считать, что высокий доход включает категории А и В, срединий - С, низкий - D и Е.

In [64]:
def income_category(row):
    income = row['total_income_category']
    
    if 'A' <= income <= 'B':
        return 'высокий доход'
    if income == 'C':
        return 'средний доход'
    if 'D' <= income <= 'E':
        return 'низкий доход'
    
data['income_category'] = data.apply(income_category, axis = 1)
data['income_category'].value_counts()

средний доход    15921
высокий доход     5039
низкий доход       371
Name: income_category, dtype: int64

Построим сводную таблицую. За индекс возьмем столбец 'debt' — имел ли задолженность по возврату кредитов, в columns определяем сегментированные значения в столбце 'income_category', в values берем значения из столбца 'total_income' и считаем кол-во.

In [65]:
income_category_pivot = data.pivot_table(index=['debt'], columns='income_category', values='total_income', aggfunc='count')
income_category_pivot

income_category,высокий доход,низкий доход,средний доход
debt,,,
0,4683,348,14568
1,356,23,1353


Далее считаем конверсию - отношение кол-ва должников к общему кол-ву клиентов по столбцу 'debt'. Для этого группируем данные по столбцу 'income_category' и считаем сумму и кол-во должников. Делим объкты друг на друга, получаем конверсию и сортируем результаты по убыванию.

In [66]:
income_category_pivot = pd.DataFrame()
income_category_pivot['sum'] = data.groupby('income_category')['debt'].sum()
income_category_pivot['count'] = data.groupby('income_category')['debt'].count()
income_category_pivot['conversion per'] = income_category_pivot['sum'] / income_category_pivot['count'] *100 
income_category_pivot.sort_values('conversion per' , ascending=False)

,sum,count,conversion per
income_category,,,
средний доход,1353,15921,8.498210
высокий доход,356,5039,7.064894
низкий доход,23,371,6.199461


**Вывод:** 

Из полученных данных можно сделать вывод, что уровень дохода также влияет на кредитную задожность. Наибольшая вероятность стать должником у клиентов со средним уровнем дохода. Самое маленькое кол-во клиентов с низким доходом, можно предположить, что они не склонны вовсе оформлять кредит. 

#### 3.4 Как разные цели кредита влияют на его возврат в срок?

Построим сводную таблицу и посчитаем отношение в процентах.

In [31]:
purpose_pivot = data.pivot_table(index = ['purpose_category'], values ='debt', aggfunc = 'count')
purpose_pivot

,debt
purpose_category,
операции с автомобилем,4279
операции с недвижимостью,10751
получение образования,3988
проведение свадьбы,2313


In [32]:
purpose_pivot  = pd.DataFrame()
purpose_pivot ['sum'] = data.groupby('purpose_category')['debt'].sum()
purpose_pivot ['count'] = data.groupby('purpose_category')['debt'].count()
purpose_pivot ['conversion per'] = purpose_pivot['sum'] / purpose_pivot ['count'] *100 
purpose_pivot .sort_values('conversion per' , ascending=False)

,sum,count,conversion per
purpose_category,,,
операции с автомобилем,400,4279,9.347978
получение образования,369,3988,9.252758
проведение свадьбы,183,2313,7.911803
операции с недвижимостью,780,10751,7.255139


**Вывод:** 

Самые опасные цель получения кредита - автомобиль и образование. Напротив, выгоднее давать кредит на свадьбу и недвижимость.

#### 3.5 Приведите возможные причины появления пропусков в исходных данных.

*Ответ:* 
Одной из частых причин таких ошибок — человеческий фактор, но также бывают и технические причины.
Пропуски в данных могут появиться разными путями. Возможно ситема сбора информации дала сбой или клиент не захотел оставлять свои данные, также пропуски оставляют умышленно, рассчитывая на автозаполнение спецсимволами.

В данном датафрейме пропуски были в столбцах days_employed — общий трудовой стаж в днях и total_income — ежемесячный доход. И составляли одинаковое кол-во значений (2174). Делаем вывод, что эти параметры связаны между собой, так как можно предположить, что на уровень дохода влияет стаж работы. И если у человека нет стажа, то он не работает и не получает доход. 

#### 3.6 Объясните, почему заполнить пропуски медианным значением — лучшее решение для количественных переменных.

*Ответ:* 
Количественные перемнные — любое числовое значение в диапазоне и их важная особенность - возможность сравнения. Для заполнения пропусков в количественных переменных лучше брать медианнон значение, так как среднее значение некорректно характеризует данные, когда некоторые значения сильно выделяются среди большинства. А медиана дает более объективные значения, так как это число, стоящее посередине упорядоченного по ряда чисел, особенно это касается данных о доходах. 

### Шаг 4: общий вывод.

Входные данные для анализа были получены от банка и включали статистику о платёжеспособности клиентов.

На первом этапе был произведен обзор данных, который выявил необходимость переработки этих данных. На этапе переработки были произведены следующие действия: поиск пропущенных и аномальных значений, поиск и обработка дубликатов. В результате были выявлены пропуски в двух столбцах days_employed и total_income. Пропущенные значения в данных столбцах были заполнен медианным значениями по каждому типу из столбца income_type. 

Далее были выявлены и обработаны аномальные значения в столбцах days_employed и children. Также для корректного анализа необходимо было заменить вещественный тип данных в столбце total_income на целочисленный. Кроме того, были обработаны неявные дубликаты в столбце education и удалены строки-дубликаты о всем датасете. 

На следующем этапе было необходимо провести категоризацию данных для более облегчения анализа и решения поставленной задачи. Так был создан столбец total_income_category с категориями:
0–30000 — 'E'; 30001–50000 — 'D'; 50001–200000 — 'C'; 200001–1000000 — 'B'; 1000001 и выше — 'A'. А также столбец purpose_category, куда вошли категории из столбца purpose: 'операции с автомобилем', 'операции с недвижимостью', 'проведение свадьбы','получение образования'. 

В результате был получен датасет с готовыми данным для анализа. Для поиска зависимостей были построены сводные таблицы и проанализированы полученные данные. 

Заключительный этап включал анализ данных, который помог выявить следующие зависимости: 
1. У клиентов без детей немного меньше вероятность стать должниками, чем у клиентов с детьми. Самая большая вероятность наличия задолженности по возврату кредитов у клиентов с 1 и 2 детьми. Но стоит отметить маленькую выборку по клиентам с более чем 2-мя детьми. 
2. Вдова/вдовец и клиенты в разводе имеет наименьшую возможность стать задолжником по кредиту. А самой опасной в этом ключе категория клиентов - неженатые и состоящие в гражданском браке.
3. Уровень дохода также влияет на кредитную задожность. Наибольшая вероятность стать должником у клиентов со средним уровнем дохода. Самое маленькое кол-во клиентов с низким доходом, можно предположить, что они не склонны вовсе оформлять кредит.
4. Самые опасные цель получения кредита - автомобиль и образование. Напротив, выгоднее давать кредит на свадьбу и недвижимость.

В результате исследования можно сделать вывод, что идеальным заемщиком является человек без детей, состоявший или состоящий в официальных отношениях, имеющий уровень дохода выше среднего и берущий деньги на недвижимость.